# Notebook 4 - Final Prediction and Submission Generation

## Goal
Generate predictions for all test pairs and create the `submission.csv` file.

## Overview
This notebook loads pre-trained models, reconstructs features for test data, generates compatibility predictions, and outputs a submission file in the required format. It leverages a blended approach for robust predictions.

## Files and Models Loaded
We load the following assets, previously trained and saved:
- `catboost_model.cbm`: A trained CatBoostRegressor model for compatibility prediction.
- `feature_columns.pkl`: A list of feature names used during CatBoost training, ensuring consistent feature ordering.
- `svd_pred_matrix.pkl`: A Singular Value Decomposition (SVD)-based compatibility matrix, providing a baseline or complementary prediction.
- `svd_user2idx.pkl`: A mapping from user IDs to SVD matrix indices.
- `train.xlsx`, `test.xlsx`, `target.csv`: Original dataset files for profile information and target pairs.

## Feature Recreation for Test Pairs
The `build_features` function is crucial. It reconstructs the same set of features for each `(src_user_id, dst_user_id)` pair in the `target.csv` as used during model training. These features include:
- **Basic numeric differences**: e.g., `age_diff`, `company_size_diff`.
- **Categorical matches**: e.g., `gender_match`, `role_match`, `industry_match`, `city_match`.
- **SVD score**: An initial compatibility estimate from the SVD model.
- **Raw categorical features**: e.g., `src_role`, `dst_industry`, etc., allowing CatBoost to directly utilize their predictive power.

Missing values in profile data are imputed using medians for numeric and 'unknown' for categorical features, mirroring the training preprocessing.

## Prediction Generation
Predictions are generated in two steps:
1. **SVD-based Prediction**: The `svd_pred_matrix.pkl` provides a base compatibility score for each user pair, captured as `svd_score` in the features.
2. **CatBoost Prediction**: The pre-trained `CatBoostRegressor` model then uses the rich set of engineered features, including the SVD score, to produce the final `compatibility_score`. Predictions are clipped to a valid range of [0, 1].

## Robustness through Blending (CatBoost + SVD)
The integration of an SVD-based `svd_score` as a feature into the CatBoost model is a form of model blending. This approach enhances robustness by:
- **Leveraging latent factors**: SVD captures underlying patterns and relationships between users that might not be explicitly present in the handcrafted features.
- **Reducing overfitting**: CatBoost can learn to weigh the SVD insights appropriately alongside other features, leading to more generalized and stable predictions.
- **Improved cold-start handling**: While CatBoost relies on explicit features, SVD can sometimes provide reasonable estimates even with sparse interaction data, contributing to overall model stability.

## Real-World Deployment Practicality
This inference pipeline is designed for practicality in a real-world scenario:
- **Pre-trained models**: Avoids retraining, making inference fast and resource-efficient.
- **Modular feature engineering**: The `build_features` function can be easily integrated into a production system to generate features for new user pairs on demand.
- **Scalable prediction**: CatBoost is optimized for fast inference, and SVD matrix lookups are efficient.
- **Clear data dependencies**: Explicitly loads required data and models, ensuring reproducibility and maintainability.

## Final Output: `submission.csv` Format
The notebook concludes by generating `submission.csv`, a CSV file with two columns:
- `ID`: A unique identifier for each user pair, formatted as `src_user_id_dst_user_id` (e.g., `5001_5002`).
- `compatibility_score`: The predicted compatibility score, a float between 0 and 1, for the corresponding user pair.

In [ ]:
!pip -q install catboost

import numpy as np
import pandas as pd
import joblib
import os

from tqdm import tqdm
from catboost import CatBoostRegressor, Pool


In [ ]:
files_needed = [
    "train.xlsx",
    "test.xlsx",
    "target.csv",
    "catboost_model.cbm",
    "feature_columns.pkl",
    "svd_pred_matrix.pkl",
    "svd_user2idx.pkl"
]

for f in files_needed:
    print(f, "->", os.path.exists(f))



train.xlsx -> True
test.xlsx -> True
target.csv -> True
catboost_model.cbm -> True
feature_columns.pkl -> True
svd_pred_matrix.pkl -> True
svd_user2idx.pkl -> True


In [ ]:
train = pd.read_excel("train.xlsx")
test  = pd.read_excel("test.xlsx")
target = pd.read_csv("target.csv")

print("Train:", train.shape)
print("Test :", test.shape)
print("Target:", target.shape)




Train: (600, 12)
Test : (400, 12)
Target: (360000, 3)


In [ ]:
model = CatBoostRegressor()
model.load_model("catboost_model.cbm")

feature_cols = joblib.load("feature_columns.pkl")

print("Loaded model + feature columns")
print("Feature columns count:", len(feature_cols))


Loaded model + feature columns
Feature columns count: 21


In [ ]:
R_hat = joblib.load("svd_pred_matrix.pkl")
user2idx = joblib.load("svd_user2idx.pkl")

print("Loaded SVD matrix:", R_hat.shape)
print("Loaded user2idx size:", len(user2idx))



Loaded SVD matrix: (600, 600)
Loaded user2idx size: 600


In [ ]:
def predict_pair(src_id, dst_id):
    if src_id not in user2idx or dst_id not in user2idx:
        return 0.5
    return float(R_hat[user2idx[src_id], user2idx[dst_id]])



In [ ]:
all_profiles = pd.concat([train, test], axis=0).reset_index(drop=True)

# Fill missing in profile table
for col in all_profiles.columns:
    if all_profiles[col].dtype == "object":
        all_profiles[col] = all_profiles[col].fillna("unknown")
    else:
        all_profiles[col] = all_profiles[col].fillna(all_profiles[col].median())

profile_dict = all_profiles.set_index("Profile_ID").to_dict(orient="index")



In [ ]:
def build_features(src_id, dst_id):
    src = profile_dict.get(src_id, None)
    dst = profile_dict.get(dst_id, None)

    if src is None or dst is None:
        return None

    feat = {}

    # Basic numeric
    feat["age_diff"] = abs(float(src["Age"]) - float(dst["Age"]))

    # Categorical matches
    feat["gender_match"] = int(str(src["Gender"]) == str(dst["Gender"]))
    feat["role_match"] = int(str(src["Role"]) == str(dst["Role"]))
    feat["seniority_match"] = int(str(src["Seniority_Level"]) == str(dst["Seniority_Level"]))
    feat["industry_match"] = int(str(src["Industry"]) == str(dst["Industry"]))
    feat["city_match"] = int(str(src["Location_City"]) == str(dst["Location_City"]))

    # Same company
    feat["same_company"] = int(str(src["Company_Name"]) == str(dst["Company_Name"]))

    # Company size diff
    try:
        feat["company_size_diff"] = abs(float(src["Company_Size_Employees"]) - float(dst["Company_Size_Employees"]))
    except:
        feat["company_size_diff"] = 0.0

    # SVD score
    feat["svd_score"] = predict_pair(src_id, dst_id)

    # Add raw categorical columns too (CatBoost likes them)
    feat["src_role"] = str(src["Role"])
    feat["dst_role"] = str(dst["Role"])
    feat["src_industry"] = str(src["Industry"])
    feat["dst_industry"] = str(dst["Industry"])
    feat["src_city"] = str(src["Location_City"])
    feat["dst_city"] = str(dst["Location_City"])

    return feat


In [ ]:
rows = []

for i in tqdm(range(len(target))):
    src_id = target.loc[i, "src_user_id"]
    dst_id = target.loc[i, "dst_user_id"]

    feat = build_features(src_id, dst_id)
    rows.append(feat)

X_pred = pd.DataFrame(rows)
print("X_pred:", X_pred.shape)
X_pred.head()



100%|██████████| 360000/360000 [00:09<00:00, 37228.52it/s]


X_pred: (360000, 15)


,age_diff,gender_match,role_match,seniority_match,industry_match,city_match,same_company,company_size_diff,svd_score,src_role,dst_role,src_industry,dst_industry,src_city,dst_city
0,0.0,1,1,1,1,1,1,0.0,0.503145,Investment Analyst,Investment Analyst,SaaS,SaaS,Sydney,Sydney
1,2.0,0,0,0,0,0,0,134.0,0.143533,Investment Analyst,Marketing Manager,SaaS,Supply Chain,Sydney,New York
2,27.0,1,0,1,0,0,0,71039.0,0.067436,Investment Analyst,Consultant,SaaS,EdTech,Sydney,Dubai
3,22.0,0,0,1,0,0,0,106.0,0.009878,Investment Analyst,Marketing Manager,SaaS,AI,Sydney,San Francisco
4,11.0,1,0,0,0,0,0,0.0,0.095487,Investment Analyst,Founder,SaaS,Climate Tech,Sydney,Berlin


In [ ]:
X_pred = X_pred.reindex(columns=feature_cols)

print("After reindex:", X_pred.shape)


After reindex: (360000, 21)


In [ ]:
force_cat_cols = [
    "src_interests","dst_interests",
    "src_objectives","dst_objectives",
    "src_constraints","dst_constraints",
    "src_role","dst_role",
    "src_industry","dst_industry",
    "src_city","dst_city"
]

for col in force_cat_cols:
    if col in X_pred.columns:
        X_pred[col] = X_pred[col].fillna("unknown").astype(str)



In [ ]:
numeric_cols = [
    "age_diff",
    "company_size_diff",
    "same_company",
    "gender_match",
    "role_match",
    "seniority_match",
    "industry_match",
    "city_match",
    "svd_score"
]

for col in numeric_cols:
    if col in X_pred.columns:
        X_pred[col] = pd.to_numeric(X_pred[col], errors="coerce")
        X_pred[col] = X_pred[col].fillna(X_pred[col].median())


In [ ]:
cat_features = [i for i, c in enumerate(X_pred.columns) if X_pred[c].dtype == "object"]

pred_pool = Pool(X_pred, cat_features=cat_features)

cat_pred = model.predict(pred_pool)
cat_pred = np.clip(cat_pred, 0, 1)

print(cat_pred[:10])


[0.21681804 0.02402364 0.01900449 0.02359358 0.02308473 0.02112475
 0.02273413 0.07628086 0.07808758 0.02339227]


In [ ]:
submission = pd.DataFrame({
    "ID": target["src_user_id"].astype(str) + "_" + target["dst_user_id"].astype(str),
    "compatibility_score": cat_pred
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv ")
submission.head()


Saved submission.csv 


,ID,compatibility_score
0,5001_5001,0.216818
1,5001_5002,0.024024
2,5001_5003,0.019004
3,5001_5004,0.023594
4,5001_5005,0.023085


In [ ]:
from google.colab import files
files.download("submission.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>